## Author Name Clean

In [10]:
# Cell 1: 导入 + 路径配置
import pandas as pd

import os
import re
import logging
from typing import Dict, List, Tuple, Optional, Set
from pathlib import Path

# ============================================================
# ✅ 路径配置 (Mac)
# ============================================================
INTERMEDIATE = "/Users/urbana-lab/Desktop/Preprocessing/data/intermediate"

INPUT_FILE           = f"{INTERMEDIATE}/merged_full_2875_with_aff.csv"         # 2875输入
CLEAN_FILE           = f"{INTERMEDIATE}/output_final_2875.csv"
OUTPUT_FILE          = f"{INTERMEDIATE}/output_final_standardized.csv"
COMPARE_FILE         = f"{INTERMEDIATE}/output_final_comparison.csv"

# 中文姓氏文件
CHINESE_SURNAME_FILE = "/Users/urbana-lab/Desktop/名词补充/EPB引文网络/data/reference/Chinese_Family_Name.xlsx"

BASE_MAPPING_FILE    = f"{INTERMEDIATE}/base_mapping.json"
AI_DECISIONS_FILE    = f"{INTERMEDIATE}/ai_decisions.json"
AI_DECISIONS_PKL     = f"{INTERMEDIATE}/ai_decisions.pkl"
UPDATED_MAPPING_FILE = f"{INTERMEDIATE}/updated_mapping.json"
CONFLICTED_FILE      = f"{INTERMEDIATE}/conflicted_cases.pkl"
FINAL_MAPPING_FILE   = f"{INTERMEDIATE}/final_mapping.json"

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

# ============================================================
# ✅ 文件检查
# ============================================================
print(f"📂 当前工作目录: {os.getcwd()}\n")
print("文件检查:")

if os.path.exists(INPUT_FILE):
    size = os.path.getsize(INPUT_FILE) / 1024 / 1024
    print(f"  ✅ 输入文件: {INPUT_FILE} ({size:.1f} MB)")
else:
    print(f"  ❌ 未找到输入文件: {INPUT_FILE}")

if os.path.exists(CHINESE_SURNAME_FILE):
    print(f"  ✅ 中文姓氏文件: 找到")
else:
    print(f"  ❌ 中文姓氏文件未找到: {CHINESE_SURNAME_FILE}")

print(f"\n✅ Cell 1 完成")

📂 当前工作目录: /Users/urbana-lab/Desktop/Preprocessing/notebooks

文件检查:
  ✅ 输入文件: /Users/urbana-lab/Desktop/Preprocessing/data/intermediate/merged_full_2875_with_aff.csv (4.1 MB)
  ✅ 中文姓氏文件: 找到

✅ Cell 1 完成


In [11]:
# Cell 3: 核心类 AuthorName + NameStandardizer + split_author_string
from dataclasses import dataclass   # ← 补这一行
@dataclass
class AuthorName:
    original: str
    last_name: str
    first_names: List[str]
    is_full_name: bool
    initials: List[str]

    def __hash__(self):
        return hash(self.original)
    def __eq__(self, other):
        return isinstance(other, AuthorName) and self.original == other.original


def split_author_string(author_string):
    """分割作者字符串"""
    if not isinstance(author_string, str):
        return []
    if ';' in author_string:
        authors = [a.strip() for a in author_string.split(';')]
    elif '|' in author_string:
        authors = [a.strip() for a in author_string.split('|')]
    elif ' and ' in author_string:
        authors = [a.strip() for a in author_string.split(' and ')]
    elif ' & ' in author_string:
        authors = [a.strip() for a in author_string.split(' & ')]
    else:
        authors = [author_string.strip()]
    return [a for a in authors if a]


class NameStandardizer:
    def __init__(self):
        self.parsed_names = {}
        self.ambiguous_cases = []
        self.common_abbreviations = {
            'Michael': ['M', 'Mike', 'Mick'], 'Robert': ['R', 'Bob', 'Rob'],
            'William': ['W', 'Bill', 'Will'], 'James': ['J', 'Jim', 'Jimmy'],
            'David': ['D', 'Dave'], 'Richard': ['R', 'Rick', 'Dick'],
            'Thomas': ['T', 'Tom', 'Tommy'], 'Christopher': ['C', 'Chris'],
            'Daniel': ['D', 'Dan', 'Danny'], 'Matthew': ['M', 'Matt'],
            'Peter': ['P', 'Pete'], 'Jonathan': ['J', 'Jon'], 'Benjamin': ['B', 'Ben'],
        }
        self.chinese_surnames_pinyin = self._load_chinese_surnames()

    def parse_author_name(self, name_str):
        if not isinstance(name_str, str):
            return None
        if name_str in self.parsed_names:
            return self.parsed_names[name_str]
        try:
            parsed_result = self._parse_name_flexible(name_str)
            if not parsed_result:
                return None
            last_name, first_names, _ = parsed_result
            if not first_names:
                return None
            is_full_name = any(len(n) > 1 for n in first_names)
            initials = [n[0].upper() for n in first_names if n]
            an = AuthorName(name_str, last_name, first_names, is_full_name, initials)
            self.parsed_names[name_str] = an
            return an
        except Exception as e:
            logger.warning(f"解析失败: {name_str}, {e}")
            return None

    def _parse_name_flexible(self, name_str):
        name_str = name_str.strip()
        if ',' in name_str:
            parts = name_str.split(',', 1)
            last_name = parts[0].strip()
            first_part = parts[1].strip()
            comps = re.findall(r'[A-Za-z]+\.?', first_part)
            first_names = [c.replace('.', '').strip() for c in comps if c.strip()]
            return last_name, first_names, "Last,First"
        elif ' ' in name_str:
            return self._parse_first_last_format(name_str)
        else:
            return None

    def _parse_first_last_format(self, name_str):
        parts = name_str.split()
        if len(parts) < 2:
            return None
        if len(parts[0]) == 1:
            return ' '.join(parts[1:]), [parts[0]], "First Last"
        if len(parts) == 2:
            return parts[1], [parts[0]], "First Last"
        elif len(parts) >= 3:
            return parts[-1], parts[:-1], "First Middle Last"
        return None

    def _detect_name_order_swap(self, name1, name2):
        if (name1.last_name.lower() in [n.lower() for n in name2.first_names] and
            name2.last_name.lower() in [n.lower() for n in name1.first_names]):
            a1 = set([name1.last_name.lower()] + [n.lower() for n in name1.first_names])
            a2 = set([name2.last_name.lower()] + [n.lower() for n in name2.first_names])
            if a1 == a2:
                return True
        return False

    def _are_different_chinese_names(self, name1, name2):
        if not (self._is_chinese_surname(name1.last_name) and self._is_chinese_surname(name2.last_name)):
            return False
        if name1.last_name.lower() != name2.last_name.lower():
            return False
        if name1.is_full_name and name2.is_full_name:
            if (len(name1.first_names) == 1 and len(name2.first_names) == 1 and
                len(name1.first_names[0]) >= 2 and len(name2.first_names[0]) >= 2 and
                name1.first_names[0].lower() != name2.first_names[0].lower()):
                return True
        elif name1.is_full_name != name2.is_full_name:
            full = name1 if name1.is_full_name else name2
            init = name2 if name1.is_full_name else name1
            if len(full.first_names) == 1:
                ffn = full.first_names[0].lower()
                iparts = []
                for x in init.first_names:
                    p = x.replace(' ', '').replace('.', '')
                    if p:
                        iparts.extend(list(p.lower()))
                if len(iparts) >= 2:
                    return True
                elif len(iparts) == 1 and ffn[0] != iparts[0]:
                    return True
                else:
                    return False
        return False

    def _load_chinese_surnames(self):
        try:
            try:
                from pypinyin import lazy_pinyin, Style
                use_pypinyin = True
            except ImportError:
                use_pypinyin = False
                logger.warning("pypinyin未安装")

            df = pd.read_excel(CHINESE_SURNAME_FILE)   # ← 已改为你的路径

            if use_pypinyin:
                surnames = df.iloc[:, 0].dropna().tolist()
                pinyin_set = set()
                for s in surnames:
                    if pd.isna(s) or not isinstance(s, str):
                        continue
                    s = str(s).strip()
                    if not s:
                        continue
                    py = ''.join(w.capitalize() for w in lazy_pinyin(s, style=Style.NORMAL))
                    pinyin_set.add(py)
                logger.warning(f"pypinyin转换 {len(pinyin_set)} 个中文姓氏")
                return pinyin_set
            return set()
        except Exception as e:
            logger.error(f"加载中文姓氏失败: {e}")
            return set()

    def _is_chinese_surname(self, surname):
        return surname in self.chinese_surnames_pinyin

    def is_same_author_hard_rules(self, name1, name2):
        if self._detect_name_order_swap(name1, name2):
            return True
        if name1.last_name.lower() != name2.last_name.lower():
            return False
        if self._are_different_chinese_names(name1, name2):
            return False
        if name1.original == name2.original:
            return True
        if name1.is_full_name and name2.is_full_name:
            if name1.first_names == name2.first_names:
                return True
            if name1.initials and name2.initials:
                if name1.initials[0].upper() != name2.initials[0].upper():
                    return False
            f1 = name1.first_names[0] if name1.first_names else ""
            f2 = name2.first_names[0] if name2.first_names else ""
            if len(f1) >= 2 and len(f2) >= 2 and f1.lower() != f2.lower():
                if self._could_be_english_nicknames(f1, f2):
                    return None
                return False
            if min(len(f1), len(f2)) == 1:
                return None
            return False
        elif name1.is_full_name != name2.is_full_name:
            if name1.initials and name2.initials:
                if name1.initials[0].upper() != name2.initials[0].upper():
                    return False
                return None
            return None
        elif not name1.is_full_name and not name2.is_full_name:
            if name1.first_names == name2.first_names:
                return True
            if self._is_initial_subset(name1, name2):
                return True
            return False
        return None

    def _is_initial_subset(self, name1, name2):
        i1 = [n.upper() for n in name1.first_names if len(n) == 1]
        i2 = [n.upper() for n in name2.first_names if len(n) == 1]
        if len(i1) != len(name1.first_names) or len(i2) != len(name2.first_names):
            return False
        if len(i1) < len(i2) and i2[:len(i1)] == i1:
            return True
        elif len(i2) < len(i1) and i1[:len(i2)] == i2:
            return True
        return False

    def _could_be_english_nicknames(self, name1, name2):
        n1, n2 = name1.lower(), name2.lower()
        for full, nicks in self.common_abbreviations.items():
            fl = full.lower()
            nl = [n.lower() for n in nicks]
            if (n1 == fl and n2 in nl) or (n2 == fl and n1 in nl):
                return True
        return False

    def build_author_groups(self, author_names):
        parsed_names = []
        for s in author_names:
            p = self.parse_author_name(s)
            if p:
                parsed_names.append(p)
        logger.warning(f"解析 {len(parsed_names)}/{len(author_names)} 个姓名")

        by_lastname = defaultdict(list)
        for n in parsed_names:
            by_lastname[n.last_name.lower()].append(n)

        result = {}
        for lastname, names in by_lastname.items():
            full_names = [n for n in names if n.is_full_name]
            initials = [n for n in names if not n.is_full_name]
            groups = {}
            for name in full_names:
                found = None
                for canon, grp in groups.items():
                    cp = self.parse_author_name(canon)
                    if cp and cp.first_names == name.first_names:
                        grp.add(name.original); found = True; break
                if not found:
                    groups[name.original] = {name.original}
            for init in initials:
                matched = False
                for canon, grp in groups.items():
                    cp = self.parse_author_name(canon)
                    if cp and self.is_same_author_hard_rules(cp, init) is True:
                        grp.add(init.original); matched = True; break
                if not matched:
                    found_i = None
                    for canon, grp in groups.items():
                        cp = self.parse_author_name(canon)
                        if cp and not cp.is_full_name and cp.first_names == init.first_names:
                            grp.add(init.original); found_i = True; break
                    if not found_i:
                        groups[init.original] = {init.original}
            result.update(groups)

        self.ambiguous_cases = []
        for i, n1 in enumerate(parsed_names):
            for n2 in parsed_names[i+1:]:
                if n1.last_name.lower() == n2.last_name.lower():
                    if self.is_same_author_hard_rules(n1, n2) is None:
                        self.ambiguous_cases.append((n1.original, n2.original))

        logger.warning(f"{len(result)} 组, {len(self.ambiguous_cases)} 个AI案例")
        return result

    def get_ai_assistance_cases(self):
        return self.ambiguous_cases

    def export_mapping(self, author_groups, filename):
        mapping = {c: list(v) for c, v in author_groups.items()}
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(mapping, f, ensure_ascii=False, indent=2)
        total = sum(len(v) for v in mapping.values())
        multi = sum(1 for v in mapping.values() if len(v) > 1)
        print(f"\n=== 映射表统计 ===")
        print(f"标准化后作者数: {len(mapping):,}")
        print(f"原始变体总数: {total:,}")
        print(f"多变体作者: {multi:,}")
        print(f"压缩率: {((total - len(mapping)) / total * 100):.2f}%")
        return mapping

print("✅ Cell 3 完成:类已定义")


✅ Cell 3 完成:类已定义


In [12]:
# Cell 4: 执行标准化
import time

# 1. 读入(用补全affiliation的文件)
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
print(f"论文数: {len(df)}")

# 2. 提取所有作者(拆分 + 去重)
all_authors = []
for authors_str in df['authors'].dropna():
    all_authors.extend(split_author_string(authors_str))

print(f"作者出现总次数: {len(all_authors):,}")

unique_authors = list(set(all_authors))
print(f"去重后唯一作者字符串: {len(unique_authors):,}")

# 3. 建分组
print(f"\n开始分组(O(n²),请耐心)...")
t0 = time.time()

standardizer = NameStandardizer()
author_groups = standardizer.build_author_groups(unique_authors)

print(f"\n耗时: {(time.time()-t0)/60:.1f} 分钟")

# 4. 导出基础映射 + 统计
mapping = standardizer.export_mapping(author_groups, BASE_MAPPING_FILE)

# 5. AI待定案例
ai_cases = standardizer.get_ai_assistance_cases()
print(f"\n需AI判断的模糊案例: {len(ai_cases):,}")
print(f"基础映射已存: {BASE_MAPPING_FILE}")

print(f"\n✅ Cell 4 完成")


论文数: 2875
作者出现总次数: 7,451
去重后唯一作者字符串: 5,661

开始分组(O(n²),请耐心)...



耗时: 0.0 分钟

=== 映射表统计 ===
标准化后作者数: 5,583
原始变体总数: 5,659
多变体作者: 76
压缩率: 1.34%

需AI判断的模糊案例: 286
基础映射已存: /Users/urbana-lab/Desktop/Preprocessing/data/intermediate/base_mapping.json

✅ Cell 4 完成


In [13]:
# Cell 5a: 测试中转API
from openai import OpenAI

client = OpenAI(
    api_key="sk-u2HoMhvbmENHNQnw7fA3297825Ba452eB5948c0101B47bB5",
    base_url="https://api.vveai.com/v1"
)

raw = client.chat.completions.with_raw_response.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "请回复:连接成功"}],
)

print("HTTP状态码:", raw.status_code)
print("原始返回:")
print(raw.text)


HTTP状态码: 200
原始返回:
{"choices":[{"finish_reason":"stop","index":0,"logprobs":null,"message":{"annotations":[],"content":"连接成功！请问有什么可以帮助您的？","refusal":null,"role":"assistant"}}],"created":1781530738,"id":"chatcmpl-Dr1ogTGzYUq7nAlrOD1PWd653EXg1","model":"gpt-4o-2024-08-06","object":"chat.completion","service_tier":"default","system_fingerprint":"fp_b08273328d","usage":{"completion_tokens":11,"completion_tokens_details":{"accepted_prediction_tokens":0,"audio_tokens":0,"reasoning_tokens":0,"rejected_prediction_tokens":0},"latency_checkpoint":{"engine_tbt_ms":11,"engine_ttft_ms":45,"engine_ttlt_ms":164,"pre_inference_ms":71,"service_tbt_ms":11,"service_ttft_ms":160,"service_ttlt_ms":274,"total_duration_ms":213,"user_visible_ttft_ms":89},"prompt_tokens":12,"prompt_tokens_details":{"audio_tokens":0,"cached_tokens":0},"total_tokens":23}}



In [15]:
# Cell 6: 跑全部286个案例
import json
import time

ai_cases = standardizer.get_ai_assistance_cases()
print(f"开始处理全部 {len(ai_cases)} 个案例...\n")

results = []  # 存所有结果

for i, (name1, name2) in enumerate(ai_cases, 1):
    # 带重试的判断
    result = None
    for attempt in range(3):  # 最多试3次
        try:
            result = judge_same_author(name1, name2)
            break
        except Exception as e:
            print(f"  ⚠️ 第{i}个出错(第{attempt+1}次尝试): {e}")
            time.sleep(2)

    if result is None:
        result = "ERROR"  # 3次都失败,标记后继续

    results.append({
        "name1": name1,
        "name2": name2,
        "judgment": result
    })

    # 进度显示(每20个报一次)
    if i % 20 == 0:
        print(f"  进度: {i}/{len(ai_cases)}")

    time.sleep(0.3)  # 限速

print("\n✅ 全部处理完成!")

# ===== 保存结果 =====
output_path = "/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/ai_judgments.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"💾 已保存到: {output_path}")

# ===== 分类统计 =====
from collections import Counter
counts = Counter(r["judgment"] for r in results)
print("\n=== 判断结果统计 ===")
for k, v in counts.items():
    print(f"  {k}: {v} 个")


开始处理全部 286 个案例...

  进度: 20/286
  进度: 40/286
  进度: 60/286
  进度: 80/286
  进度: 100/286
  进度: 120/286
  进度: 140/286
  进度: 160/286
  进度: 180/286
  进度: 200/286
  进度: 220/286
  进度: 240/286
  进度: 260/286
  进度: 280/286

✅ 全部处理完成!
💾 已保存到: /Users/urbana-lab/Desktop/Preprocessing/data/intermediate/ai_judgments.json

=== 判断结果统计 ===
  YES: 275 个
  UNSURE: 10 个
  NO: 1 个


In [16]:
# Cell 7: 查看需要人工复核的案例(UNSURE + NO)
import json

with open("/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/ai_judgments.json", encoding="utf-8") as f:
    results = json.load(f)

review = [r for r in results if r["judgment"] != "YES"]
print(f"需人工复核 {len(review)} 个:\n")

for i, r in enumerate(review, 1):
    print(f"{i:2d}. [{r['judgment']:6s}] 【{r['name1']}】 vs 【{r['name2']}】")


需人工复核 11 个:

 1. [UNSURE] 【Rao, Pooja】 vs 【Rao, P Suresh C】
 2. [UNSURE] 【Wilson, Alāna M】 vs 【Wilson, A G】
 3. [UNSURE] 【Kim, Sun-Joong】 vs 【Kim, S H】
 4. [UNSURE] 【Kim, Sun-Joong】 vs 【Kim, S Hong】
 5. [UNSURE] 【Han, Sun Sheng】 vs 【Han, S-Y】
 6. [UNSURE] 【Lee, C-M】 vs 【Lee, Chanam】
 7. [UNSURE] 【Lee, C-M】 vs 【Lee, Changyeon】
 8. [UNSURE] 【Wu, Xiao-Lei】 vs 【Wu, X Ben】
 9. [UNSURE] 【Smith, J MacGregor】 vs 【Smith, Jim】
10. [UNSURE] 【Wong, S C】 vs 【Wong, Sze-Chun】
11. [NO    ] 【Nelson, J A Montiel】 vs 【Nelson, Jake】


In [22]:
import json
import pandas as pd
from collections import defaultdict

# ===== 1. 读取 =====
with open("/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/base_mapping.json", encoding="utf-8") as f:
    base_mapping = json.load(f)  # {标准名: [变体列表]}

with open("/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/ai_judgments.json", encoding="utf-8") as f:
    ai_results = json.load(f)

# ===== 2. 反转base_mapping → {变体: 标准名} 方便查找 =====
variant_to_standard = {}
for standard, variants in base_mapping.items():
    for v in variants:
        variant_to_standard[v] = standard
    variant_to_standard[standard] = standard  # 标准名自己也映射到自己

# ===== 3. 收集YES对(275 AI + 1 手动Wong) =====
merge_pairs = [(r["name1"], r["name2"]) for r in ai_results if r["judgment"] == "YES"]
merge_pairs.append(("Wong, S C", "Wong, Sze-Chun"))
print(f"待合并对数: {len(merge_pairs)}")

# ===== 4. 并查集处理链式 =====
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    parent[find(a)] = find(b)

for n1, n2 in merge_pairs:
    union(n1, n2)

# ===== 5. 每个YES组选最长(全名)为标准 =====
groups = defaultdict(list)
for name in parent:
    groups[find(name)].append(name)

yes_mapping = {}
for root, members in groups.items():
    standard = max(members, key=len)  # 最长=全名
    for m in members:
        yes_mapping[m] = standard

# 强制Wong用全名
for m in ["Wong, S C", "Wong, Sze-Chun"]:
    if m in yes_mapping:
        yes_mapping[m] = "Wong, Sze-Chun"

# ===== 6. 最终映射函数:raw → base标准化 → YES合并 =====
def final_name(raw):
    name = variant_to_standard.get(raw, raw)  # 先base标准化
    name = yes_mapping.get(name, name)        # 再YES合并
    return name

# ===== 7. 应用 =====
df = pd.read_csv("/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/merged_full_2875_with_aff.csv")

def normalize_authors(s):
    if pd.isna(s):
        return s
    authors = [a.strip() for a in str(s).split(";")]
    return "; ".join(final_name(a) for a in authors)

df["authors_normalized"] = df["authors"].apply(normalize_authors)

# ===== 8. 保存 =====
out_path = "/Users/urbana-lab/Desktop/Preprocessing/data/intermediate/merged_full_2875_with_aff_normalized.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"💾 已保存: {out_path}")

# ===== 9. 统计 =====
raw_set, final_set = set(), set()
for s in df["authors"].dropna():
    for a in str(s).split(";"):
        raw_set.add(a.strip())
for s in df["authors_normalized"].dropna():
    for a in str(s).split(";"):
        final_set.add(a.strip())

print(f"\n=== 最终统计 ===")
print(f"标准化前唯一作者: {len(raw_set)}")
print(f"标准化后唯一作者: {len(final_set)}")
print(f"总共合并减少: {len(raw_set) - len(final_set)} 个")



待合并对数: 276
💾 已保存: /Users/urbana-lab/Desktop/Preprocessing/data/intermediate/merged_full_2875_with_aff_normalized.csv

=== 最终统计 ===
标准化前唯一作者: 5661
标准化后唯一作者: 5339
总共合并减少: 322 个
